# Build ML model for analysis and prediction of DNA-encoded library data.

In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem import Descriptors, rdPartialCharges
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

import itertools
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import pathlib
from pathlib import Path
from rdkit import RDLogger
from tqdm.notebook import tqdm

# Get the main logger
logger = RDLogger.logger()

# Set its level to ERROR so only serious messages show up
logger.setLevel(RDLogger.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#verify Cuda is actually working

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)

print("Torch CUDA libraries loaded:")
for lib in torch.cuda._CudaBase.__module__.split(":"):
    print(lib)

Torch version: 2.8.0+cu128
CUDA available: True
CUDA version (runtime): 12.8
Torch CUDA libraries loaded:
torch.cuda


## Setup.

In [2]:
readout = 'kindel_del'

In [3]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

In [4]:
import s3fs

for dataset in ['ddr1', 'mapk14']: 
    fs = s3fs.S3FileSystem(anon=True)
    info = fs.info(f'kin-del-2024/data/{dataset}.parquet')
    print(f"File size for {dataset}: {info['size']/1e6:.2f} MB")

File size for ddr1: 6782.18 MB
File size for mapk14: 6911.55 MB


In [5]:
import pyarrow.parquet as pq

def read_first_n_rows(file_path, n_rows):
    parquet_file = pq.ParquetFile(file_path)
    
    num_row_groups = parquet_file.num_row_groups
    
    rows_read = 0
    result = []

    for i in range(num_row_groups):
        row_group = parquet_file.read_row_group(i)
        df = row_group.to_pandas()

        if rows_read + len(df) >= n_rows:
            result.append(df.head(n_rows - rows_read))
            break
        
        result.append(df)
        rows_read += len(df)

    final_df = pd.concat(result, ignore_index=True).head(n_rows)
    return final_df

In [6]:
def count_rows(file_path):
    parquet_file = pq.ParquetFile(file_path)
    total_rows = sum(parquet_file.metadata.row_group(i).num_rows
                     for i in range(parquet_file.num_row_groups))
    return total_rows

In [15]:
target = f"{DATA}/ddr1.parquet"

row_count = count_rows(target)
print(f"Total rows: {row_count}")

N = 1000  # Number of rows to load - for prototyping
df = read_first_n_rows(target, N)

Total rows: 67641617


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
# import umap

# Define a function to compute Morgan fingerprints (a type of molecular fingerprint)
def compute_fingerprint(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    else:
        return None

# # Assuming your DataFrame is structured as follows:
# # df = pd.read_csv('your_data.csv')  # For example, a CSV with columns 'smiles_a', 'smiles_b', 'smiles_c', 'smiles_final'
# # You can replace this with your actual DataFrame or CSV path.
# df = pd.DataFrame({
#     'smiles_a': ['CCO', 'CCN', 'CCO', 'CNC'],
#     'smiles_b': ['CCO', 'CCO', 'CN', 'CN'],
#     'smiles_c': ['CCC', 'CNC', 'CCC', 'CCO'],
#     'smiles_final': ['CCOCCO', 'CCOCCN', 'CCOCCC', 'CNCN']
# })

# Compute the fingerprints for each synthon and the final molecule
def get_fingerprints(df):
    # Apply the fingerprint computation to each synthon and the final molecule
    df['fp_a'] = df['smiles_a'].apply(compute_fingerprint)
    df['fp_b'] = df['smiles_b'].apply(compute_fingerprint)
    df['fp_c'] = df['smiles_c'].apply(compute_fingerprint)
    df['fp_final'] = df['smiles_final'].apply(compute_fingerprint)
    return df

# Apply to your DataFrame
df = get_fingerprints(df)

# Convert fingerprints to numpy arrays (bit vectors)
fingerprints = []
for index, row in df.iterrows():
    fingerprints.append(np.concatenate([np.array(row['fp_a']), np.array(row['fp_b']), np.array(row['fp_c']), np.array(row['fp_final'])]))
    
fingerprints = np.array(fingerprints)

# Normalize the data: PCA, t-SNE, and UMAP all work better with normalized data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
fingerprints_scaled = scaler.fit_transform(fingerprints)

# ---------------------------
# Apply PCA for dimensionality reduction
pca = PCA(n_components=2)
pca_result = pca.fit_transform(fingerprints_scaled)

# Apply t-SNE for non-linear dimensionality reduction
tsne = TSNE(n_components=2, perplexity=30, n_iter=300)
tsne_result = tsne.fit_transform(fingerprints_scaled)

# # Apply UMAP for non-linear dimensionality reduction
# umap_model = umap.UMAP(n_components=2)
# umap_result = umap_model.fit_transform(fingerprints_scaled)

# ---------------------------
# Plotting the results

# Function to plot results
def plot_2d_projection(X, title="Dimensionality Reduction", x_label="Component 1", y_label="Component 2"):
    plt.figure(figsize=(8, 6))
    plt.scatter(X[:, 0], X[:, 1], alpha=0.7, edgecolors='w', s=100)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.show()

# Plot PCA result
plot_2d_projection(pca_result, title="PCA of Synthons and Final Molecules")

# Plot t-SNE result
plot_2d_projection(tsne_result, title="t-SNE of Synthons and Final Molecules")

# Plot UMAP result
# plot_2d_projection(umap_result, title="UMAP of Synthons and Final Molecules")


[16:56:31] SMILES Parse Error: syntax error while parsing: CC(C)(C)OC(=O)N1C[C@@H](C[C@H]1C(O)=O)NC(=O)OCC1c2ccccc2-c2ccccc12,
[16:56:31] SMILES Parse Error: Failed parsing SMILES 'CC(C)(C)OC(=O)N1C[C@@H](C[C@H]1C(O)=O)NC(=O)OCC1c2ccccc2-c2ccccc12,' for input: 'CC(C)(C)OC(=O)N1C[C@@H](C[C@H]1C(O)=O)NC(=O)OCC1c2ccccc2-c2ccccc12,'
[16:56:31] SMILES Parse Error: syntax error while parsing: [H][C@@](CCC(O)=O)(NC(=O)OC(C)(C)C)C(=O)OC,
[16:56:31] SMILES Parse Error: Failed parsing SMILES '[H][C@@](CCC(O)=O)(NC(=O)OC(C)(C)C)C(=O)OC,' for input: '[H][C@@](CCC(O)=O)(NC(=O)OC(C)(C)C)C(=O)OC,'
[16:56:31] SMILES Parse Error: syntax error while parsing: CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc12)C(O)=O,
[16:56:31] SMILES Parse Error: Failed parsing SMILES 'CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc12)C(O)=O,' for input: 'CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc12)C(O)=O,'
[16:56:31] SMILES Parse Error: syntax error while parsing: CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc

KeyError: 'smiles_final'